# Paired & Randomization Tests

When observations are matched, the comparison must preserve the pairs. This page explains paired tests, design-based randomization, and planning sample size and power.

## Paired Comparisons

Use pairing when the same unit receives two conditions, or when two units are meaningfully matched. Examples include two measurements on the same participant or matched specimens from the same batch.

Define a consistent subtraction order:

$$D_j=Y_{1j}-Y_{2j},\qquad j=1,\ldots,n.$$

Here **$n$ counts pairs**, not the $2n$ individual measurements. The comparison becomes a one-sample problem for the differences:

$$\bar D=\frac1n\sum_jD_j,\qquad
s_D^2=\frac{\sum_j(D_j-\bar D)^2}{n-1}.$$

To test $H_0:\mu_D=\Delta_0$,

$$T=\frac{\bar D-\Delta_0}{s_D/\sqrt n}\sim t_{n-1}\quad\text{under }H_0.$$

The exact result assumes independent, normally distributed **differences**. The two sets of raw measurements need not individually satisfy an independent-sample model. A two-sided confidence interval is

$$\bar D\pm t_{1-\alpha/2,n-1}\frac{s_D}{\sqrt n}.$$

### Why Pairing Can Improve Precision

$$\operatorname{Var}(Y_1-Y_2)=\sigma_1^2+\sigma_2^2-2\operatorname{Cov}(Y_1,Y_2).$$

When responses within pairs are positively correlated, subtracting removes shared unit-to-unit variation. With equal marginal variance $\sigma^2$ and correlation $\rho$,

$$\operatorname{Var}(\bar D)=\frac{2\sigma^2(1-\rho)}{n}.$$

- Strong positive correlation can give much better precision than an independent comparison of the same size.
- Arbitrary matching does not guarantee a gain. Negative within-pair correlation can increase variance.
- For crossover experiments, randomize or counterbalance treatment order. Time trends, learning, fatigue, and carryover can otherwise mimic treatment effects.
- If a treatment permanently changes the unit, applying both conditions may not be appropriate.

:::{dropdown} Worked Example: Six Matched Differences

Suppose the differences are $2,1,3,2,4,3$ response units.

$$\bar d=2.5,\qquad s_D^2=1.1,\qquad
\operatorname{SE}=\sqrt{1.1/6}=0.4282.$$

For $H_0:\mu_D=0$,

$$t=\frac{2.5}{0.4282}=5.8387,\qquad df=5.$$

The two-sided $p$-value is approximately $0.0021$; the 95% interval is $(1.3993,3.6007)$.

Under the paired normal-difference model, condition 1 has a higher mean response. Whether a higher response is desirable depends on the outcome. The experimental unit and pairing must be genuine for this uncertainty calculation to apply.
:::

## Randomization & Permutation Tests

A randomization test asks: **How unusual is the observed statistic among assignments allowed by the design, if treatment has no effect?**

1. Choose a statistic, such as $\bar Y_1-\bar Y_2$.
2. Compute its observed value $T_{\mathrm{obs}}$.
3. Generate treatment assignments consistent with the original randomization.
4. Recompute the statistic for each assignment.
5. Count results at least as extreme as observed.

For a complete enumeration of equally likely assignments $\mathcal A$, a two-sided absolute-statistic test has

$$p=\frac{\#\{a\in\mathcal A:|T_a|\geq|T_{\mathrm{obs}}|\}}{|\mathcal A|}.$$

For $B$ independently sampled random assignments, a standard Monte Carlo estimate is

$$\widehat p=\frac{1+\sum_{b=1}^B I(|T_b|\geq|T_{\mathrm{obs}}|)}{B+1}.$$

The added $1$ includes the observed arrangement and avoids reporting a zero estimated $p$-value.

| Design | Valid rearrangement |
|---|---|
| Completely randomized independent groups | Reassign labels while retaining the original group sizes. |
| Randomized matched pairs | Swap treatment labels **within each pair**, equivalent to sign-flipping each difference. |
| Randomized blocks | Reassign labels **within blocks**, preserving allocation counts. |
| Cluster-randomized experiment | Reassign whole clusters, not individual measurements. |

:::{note}
A design-based exact randomization test typically tests the **sharp null of no treatment effect on any unit**. A general permutation test requires exchangeability under its null. Equality of population means alone does not justify freely shuffling labels when group distributions differ. For non-randomized paired data, sign-flipping also needs suitable symmetry assumptions.
:::

A **bootstrap** resamples observations to approximate sampling uncertainty. A **permutation test** rearranges labels to construct a null distribution. These are different procedures.

:::{dropdown} Exact Paired Randomization Example

For the six differences above, suppose treatments were independently randomized within each pair. There are $2^6=64$ equally likely sign patterns under the sharp null.

All six observed differences are positive. Only the all-positive and all-negative patterns have an absolute mean as large as $2.5$:

$$p=\frac2{64}=0.03125.$$

This differs from the paired $t$ result because the two methods use different null reference distributions and assumptions. The randomization distribution is discrete; with only six pairs it has limited resolution.
:::

## Planning Sample Size & Power

Choose the **smallest effect worth detecting**, $\delta$, before data collection. Specify a significance level $\alpha$, desired power $1-\beta$, and a plausible variance from earlier work or a pilot study.

For a balanced, two-sided independent comparison with common variance $\sigma^2$, a normal approximation gives

$$n\approx\frac{2\sigma^2(z_{1-\alpha/2}+z_{1-\beta})^2}{\delta^2}
\quad\text{units per group}.$$

For paired differences with variance $\sigma_D^2$,

$$n\approx\frac{\sigma_D^2(z_{1-\alpha/2}+z_{1-\beta})^2}{\delta^2}
\quad\text{pairs}.$$

Here $z_q$ is the $q$ quantile of the standard normal distribution. Round up. These are planning approximations; small-sample $t$ procedures, attrition, clustering, and a different design may require different calculations.

| Change, holding other inputs fixed | Required sample size |
|---|---|
| Higher desired power | Increases. |
| Larger response variance | Increases. |
| Smaller significance level | Increases. |
| Smaller target effect | Increases; halving $\delta$ roughly quadruples $n$. |
| More effective pairing | Decreases when it reduces $\sigma_D^2$. |

:::{dropdown} Planning Example

Suppose $\sigma=1$, $\delta=0.5$, $\alpha=0.05$, and desired power is $0.80$.

$$n\approx\frac{2(1)^2(1.9600+0.8416)^2}{0.5^2}=62.79.$$

Plan for approximately **63 independent units per group**, or 126 total, before allowance for missing data. This is a normal-approximation target rather than an exact guarantee of 80% power for a finite-sample $t$ test.
:::

Increasing the number of correlated measurements is not the same as increasing the number of independent experimental units. Large samples do not fix confounding.

## Reproduce the Paired Examples

These calculations compare the paired normal-model test with exact within-pair sign randomization for the same illustrative differences.

In [1]:
from itertools import product
import numpy as np
from scipy import stats

# Paired normal-model test and exact within-pair sign randomization.
d = np.array([2., 1., 3., 2., 4., 3.])
paired = stats.ttest_1samp(d, 0)
signs = np.array(list(product([-1, 1], repeat=len(d))))
randomized_means = (signs * d).mean(axis=1)
p_exact = np.mean(np.abs(randomized_means) >= abs(d.mean()) - 1e-12)
print(f"Paired: t={paired.statistic:.4f}, p={paired.pvalue:.6f}")
print(f"Exact sign-flip p={p_exact:.5f}")



Paired: t=5.8387, p=0.002085
Exact sign-flip p=0.03125


## Quick Review

:::{dropdown} Self-Check
**Why can pairing improve precision?** Positive within-pair covariance reduces the variance of the difference.

**Can a blocked experiment be analyzed by shuffling every observation freely?** No. Preserve the randomization restrictions, including blocks, pairs, or whole clusters.

**Does a bootstrap construct the same reference distribution as a permutation test?** No. A bootstrap resamples observations to approximate sampling uncertainty; a permutation test rearranges labels under a null model.

**What happens when the target effect is halved?** The simple planning approximation requires about four times the independent sample size, with other inputs held fixed.
:::